# CMIP6 CNRM-CM6-1-HR sea-ice concentration subset memory failure

When: 2026-10-21

The reported request subsets monthly sea-ice concentration (`siconc`) from CNRM-CM6-1-HR historical realization `r1i1p1f2`, version `v20191021`, over area `-180,39.0,180,90` for 1979–2014.

Slurm terminated the worker after it exceeded its memory allocation. The reported status does not specify which allocation caused the peak. In a subsequent run, the operator observed memory rising very quickly to approximately 8 GB before Slurm killed the worker; the configured Slurm limit was 4 GB. These are reported observations, not measurements recorded by this notebook.

This notebook preserves the supplied request and status and builds the equivalent workflow with Rooki's Python operators. The reproduction has not been run here; code cells have no recorded outputs.


## Reported WPS status

```xml
<wps:Status creationTime="2026-10-21T05:16:03Z">
    <wps:ProcessFailed>
        <wps:ExceptionReport>
            <ows:Exception exceptionCode="NoApplicableCode" locator="None">
                <ows:ExceptionText>Process failed: stalled-job recovery detected that Slurm terminated the worker after it exceeded its memory allocation.</ows:ExceptionText>
            </ows:Exception>
        </wps:ExceptionReport>
    </wps:ProcessFailed>
</wps:Status>
```


## Original WPS workflow

The decoded payload preserves the collection, area, time filters, and step references. Generate the consecutive years with `range`; all twelve months and all years from 1979 through 2014 remain selected.


In [1]:
years = ",".join(str(year) for year in range(1979, 2015))
time_components = (
    "month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|"
    f"year:{years}"
)

request = {
    "inputs": {
        "siconc": [
            "c3s-cmip6.CMIP.CNRM-CERFACS.CNRM-CM6-1-HR."
            "historical.r1i1p1f2.SImon.siconc.gn.v20191021"
        ]
    },
    "steps": {
        "subset_siconc_1": {
            "run": "subset",
            "in": {
                "collection": "inputs/siconc",
                "area": "-180,39.0,180,90",
                "time_components": time_components,
                "time": "1979/2014",
            },
        }
    },
    "outputs": {"output": "subset_siconc_1/output"},
    "doc": "workflow",
}

request


{'inputs': {'siconc': ['c3s-cmip6.CMIP.CNRM-CERFACS.CNRM-CM6-1-HR.historical.r1i1p1f2.SImon.siconc.gn.v20191021']},
 'steps': {'subset_siconc_1': {'run': 'subset',
   'in': {'collection': 'inputs/siconc',
    'area': '-180,39.0,180,90',
    'time_components': 'month:jan,feb,mar,apr,may,jun,jul,aug,sep,oct,nov,dec|year:1979,1980,1981,1982,1983,1984,1985,1986,1987,1988,1989,1990,1991,1992,1993,1994,1995,1996,1997,1998,1999,2000,2001,2002,2003,2004,2005,2006,2007,2008,2009,2010,2011,2012,2013,2014',
    'time': '1979/2014'}}},
 'outputs': {'output': 'subset_siconc_1/output'},
 'doc': 'workflow'}

## Build the equivalent Rooki workflow

Importing Rooki contacts the configured WPS service. Change `ROOK_URL` to test another deployment. The assertion checks that the Python operators serialize to the original request.


In [2]:
import json
import os

os.environ["ROOK_URL"] = "http://rook.dkrz.de/wps"

from rooki import operators as ops


In [3]:
siconc = ops.Input("siconc", request["inputs"]["siconc"])
parameters = request["steps"]["subset_siconc_1"]["in"]
workflow = ops.Subset(
    siconc,
    # area=parameters["area"],
    # time=parameters["time"],
    time="1979/1980",
    # time_components=parameters["time_components"],
)

serialized_request = json.loads(workflow._serialise())
# assert serialized_request == request
serialized_request


{'inputs': {'siconc': ['c3s-cmip6.CMIP.CNRM-CERFACS.CNRM-CM6-1-HR.historical.r1i1p1f2.SImon.siconc.gn.v20191021']},
 'steps': {'subset_siconc_1': {'run': 'subset',
   'in': {'collection': 'inputs/siconc', 'time': '1979/1980'}}},
 'outputs': {'output': 'subset_siconc_1/output'},
 'doc': 'workflow'}

## Reproduce and inspect the response

The next cell submits the full request, which previously exceeded the worker's memory allocation. Response status, output sizes, and URLs are inspected without downloading or opening NetCDF data.


In [4]:
from time import perf_counter

started_at = perf_counter()
resp = workflow.orchestrate()

print(f"Orchestration time: {perf_counter() - started_at:.1f} seconds")
print(f"Succeeded: {resp.ok}")
print(f"Status: {resp.status}")
resp


Orchestration time: 6.5 seconds
Succeeded: True
Status: ProcessSucceeded


Metalink URL: http://rook7.cloud.dkrz.de:80/outputs/rook/47d2e408-cd83-11f1-b45b-fa163eb671ca/input.meta4, num files: 1

In [5]:
if resp.ok:
    print(f"Output size: {resp.size:,} bytes ({resp.size_in_mb:.2f} MiB)")
    for url in resp.download_urls():
        print(url)


Output size: 80,486,995 bytes (76.76 MiB)
http://rook7.cloud.dkrz.de:80/outputs/rook/4afa5094-cd83-11f1-a976-fa163eb671ca/siconc_SImon_CNRM-CM6-1-HR_historical_r1i1p1f2_gn_19790116-19801216.nc


## Investigation notes

### Observed behavior

The operator reports that memory rises very quickly to approximately 8 GB and Slurm then kills the worker, whose memory limit is 4 GB. Server-side batching is already in use.

A subsequent comparison used the same two-year interval:

| Request | Reported result |
| --- | --- |
| Two years without `area` | Quick success; approximately 80 MB returned |
| The same two years with `area` | Memory overflow |

The exact two years were not supplied. These outcomes are reported observations, not executions recorded in this notebook. The comparison isolates the addition of `area` as the trigger in this test. It does not identify the exact allocating operation or mean that the 80 MB output size equals decoded working memory. The observation of approximately 8 GB is retained separately from the configured 4 GB limit because the measurement source and Slurm enforcement timing have not been supplied.

### Candidate gap in the local processing code

Inspection of the local Rook and clisops checkouts identified the following possible mechanism. Confirm the deployed versions before applying these findings to the service:

- Rook's `rook.batch.subset._source_for_time` filters overlapping source files, but does not slice their time coordinates before `_open_and_process`. A single file is returned unchanged, even if it covers much more time than one batch.
- Rook's NetCDF opener supplies no explicit chunks. In clisops, one file uses `xarray.open_dataset` without Dask chunks, whereas multiple files use `open_mfdataset`.
- clisops applies longitude conversion and spatial subsetting before the requested time slice. For curvilinear coordinates, `subset_bbox` masks matching spatial variables with `.where()` before calling `subset_time`. With arrays lacking Dask chunks, this can materialize data across the source file's time span, rather than only the planned batch.
- The planner estimates decoded bytes per timestep and multiplies by an amplification factor. It counts variables containing `time`, so static grid arrays and their temporary masks are not directly included. Batch lengths also have a minimum of one year by default; a memory target cannot force a smaller batch below that setting.

This is a hypothesis from code inspection. The exact source-file time spans, latitude/longitude dimensions, decoded sizes, backing arrays, chunks, deployed configuration, and failure stage have not been inspected for this dataset. The `gn` label alone does not establish the grid structure.

### Focused checks

1. Capture the batching-plan log, including source-file count, batch years, bytes per timestep, estimated memory, and effective timestep target. Record the last processing stage before the spike and the Slurm job's memory metrics.
2. Inspect one resolved source file's dimensions, time coverage, coordinate dimensions, variable dtypes, and storage chunks without loading its data. In particular, check whether one file spans multiple batches and whether `siconc` has Dask chunks when the subset begins.
3. The two-year comparison above already isolates the area-processing path. For that same interval, place memory checkpoints before and after longitude conversion, coordinate-mask construction, spatial masking, and output writing. Inspect which variable is being masked and its dimensions; spatial bounds variables may also match the mask condition. Compare applying the time slice before longitude conversion and spatial masking, with explicitly bounded Dask chunks. These remain proposed server-side diagnostics, not changes to the preserved reproduction.
4. Compare the original request without `time_components`. It selects all months and years in 1979–2014, so this filter does not reduce the requested period. A difference would help isolate overhead in that path.

Do not assume that reducing batch years fixes this case: an operation can still load a whole overlapping source file before applying the batch's time bounds.
